# Task: post a theme vote

Build a new theme vote, post the poll + one comment per option to Bluesky, then save it. Headless version of the repo-root `post_theme_vote.ipynb`, run by `core.scheduler`.

In [ ]:
dry_run = False  # build the vote but skip posting and saving

In [ ]:
import os
import sys

# add the repo root to sys.path so `import config` / `from core...` resolve
# when this notebook is run directly (e.g. from VSCode); when the scheduler
# runs it via papermill the working directory is already the root, so this is
# a harmless no-op.
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'pyproject.toml')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import atproto
from atproto import models

import config
from core.database import session_scope
from core.interactions.votes import ThemeVoteGenerator, add_theme_vote

In [ ]:
vote = ThemeVoteGenerator().generate()

print('built (unsaved) vote')
print(f'  voting:       {vote.vote_start_date} -> {vote.vote_end_date}')
print(f'  theme window: {vote.theme_start_date} -> {vote.theme_end_date}')
print('  options:', [o.theme_name for o in vote.options])

In [ ]:
if dry_run:
    print('dry_run: built vote only, not posting or saving')
else:
    client = atproto.Client()
    client.login(config.ATPROTO_CLIENT_USERNAME, config.ATPROTO_CLIENT_PASSWORD)

    # the poll itself
    root = client.send_post(text=vote.post_text)
    vote.post_uri = root.uri
    vote.post_cid = root.cid
    print('posted poll:', vote.post_uri)

    # every option comment is a direct reply to the poll (siblings under the post)
    root_ref = models.create_strong_ref(root)
    for option in vote.options:
        reply_ref = models.AppBskyFeedPost.ReplyRef(parent=root_ref, root=root_ref)
        comment = client.send_post(text=option.comment_text, reply_to=reply_ref)
        option.comment_uri = comment.uri
        option.comment_cid = comment.cid
        print(f'  commented {option.theme_name}: {comment.uri}')

    with session_scope() as session:
        vote = add_theme_vote(session, vote)
    print(f'saved vote {vote.id}')